# 📘 Notebook 2c: Distributed Training (DDP)

## Overview

**Alternative to Notebook 2.** Scale a **single** DNN across multiple GPUs using native PyTorch Distributed Data Parallel (DDP), orchestrated by Snowflake's `PyTorchDistributor`. Use this when one global model is the right abstraction but you need more GPU throughput than a single node provides.

### When DDP Makes Sense

**Good candidates**
- Training time per epoch is a real bottleneck
- 2+ GPUs are available and single-GPU utilization is already near 100%
- Data loading isn't the bottleneck (DDP won't help if it is)

**Usually overkill**
- Single-GPU training is already fast
- GPU utilization is low — fix the data pipeline first
- Only 1 GPU available

> **Profile first.** Dataset size alone isn't a reliable trigger; model, batch size, and I/O pattern all matter more.

### How it Works

Each GPU gets its own copy of the model and a shard of the training data. Gradients are synchronized across GPUs via `AllReduce` every step. `PyTorchDistributor` handles process spawning, sharding, and collective init.

| Component | Purpose |
|---|---|
| `ShardedDataConnector` | Auto-partitions the dataset across workers |
| `PyTorchDistributor` | Orchestrates distributed job execution |
| `PyTorchScalingConfig` | Declares nodes, workers, and per-worker resources |
| `get_context()` | Per-worker rank / device / dataset map |

### What's Covered

| Section | Topic |
|---|---|
| 1 | Load `v1_train` + `v1_test` as `ShardedDataConnector`s |
| 2 | Define model architecture (inside the training function) |
| 3 | Write the DDP training function (LR-scaled, rank-0 eval) |
| 4 | Configure `PyTorchDistributor` and launch |
| 5 | Retrieve and register the trained model |
| 6 | Summary |

### Handoff from Notebook 1

Loads `WAFER_YIELD_TRAINING_DATASET v1_train` and `v1_test`.

### Handoff to Registry

Registers the DDP-trained model under its own name (`WAFER_YIELD_DDP_MODEL`). This is separate from Notebook 2's champion — use Notebook 2 to pick the production model.

### References

- [Snowflake PyTorchDistributor](https://docs.snowflake.com/en/developer-guide/snowpark-ml/reference/latest/container-runtime/distributors.pytorch_distributor)


In [ ]:
%%sql -r dataframe_1
# ============================================================================
# (Optional) Bump pool capacity. Leave commented — only needed if you want 3 nodes.
# ============================================================================
# session.sql("ALTER COMPUTE POOL WAFER_TRAINING_POOL SET MAX_NODES = 3").collect()


In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
session.sql("DESCRIBE COMPUTE POOL WAFER_TRAINING_POOL").show()

In [ ]:
# ============================================================================
# IMPORTS
# ============================================================================

import os
import numpy as np
import pandas as pd
from datetime import datetime

# PyTorch imports
import torch
import torch.nn as nn
import torch.optim as optim

# Snowpark imports
from snowflake.snowpark.context import get_active_session

# Snowflake ML Dataset imports
from snowflake.ml import dataset

# Snowflake ML Data imports
from snowflake.ml.data import DataConnector
from snowflake.ml.data.sharded_data_connector import ShardedDataConnector

# Snowflake Distributed Training imports
from snowflake.ml.modeling.distributors.pytorch import (
    PyTorchDistributor,
    PyTorchScalingConfig,
    WorkerResourceConfig
)

print("✅ Imports complete")
print(f"   PyTorch: {torch.__version__}")
print(f"   CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU count: {torch.cuda.device_count()}")


In [ ]:
# ============================================================================
# SNOWFLAKE SESSION
# ============================================================================

session = get_active_session()

session.sql("USE DATABASE WAFER_YIELD_DEMO").collect()
session.sql("USE SCHEMA RAW_DATA").collect()

print(f"✅ Session active")
print(f"   Database: {session.get_current_database()}")
print(f"   Schema: {session.get_current_schema()}")

---

## 📘 Section 1 — Load Data with `ShardedDataConnector`

`ShardedDataConnector` partitions the source dataset across workers automatically — each GPU worker receives a unique shard.

| Property | Why it matters |
|---|---|
| **Unique shards per worker** | Prevents duplicate gradient signals across GPUs |
| **Memory-safe** | No worker tries to hold the whole dataset |
| **Required for DDP correctness** | `AllReduce` assumes independent mini-batches per rank |
| **Auto-partitioned** | No manual splitting logic required |

This cell loads both the `v1_train` and `v1_test` splits from Notebook 1 — training runs on `train`; rank 0 evaluates on `test`.


In [ ]:
# ============================================================================
# LOAD ML DATASETS (v1_train + v1_test) AS SHARDED DATA CONNECTORS
# ============================================================================
DATASET_NAME  = "WAFER_YIELD_DEMO.RAW_DATA.WAFER_YIELD_TRAINING_DATASET"
TRAIN_VERSION = "v1_train"
TEST_VERSION  = "v1_test"

print(f"Loading datasets: {TRAIN_VERSION} + {TEST_VERSION}")
train_ds = dataset.load_dataset(session, DATASET_NAME, TRAIN_VERSION)
test_ds  = dataset.load_dataset(session, DATASET_NAME, TEST_VERSION)

sharded_train = ShardedDataConnector.from_dataset(train_ds)
sharded_test  = ShardedDataConnector.from_dataset(test_ds)

training_df = train_ds.read.to_snowpark_dataframe()
EXCLUDE_COLS = ['WAFER_ID', 'FIRST_TIMESTAMP', 'YIELD_GOOD', 'YIELD_SCORE', 'DOMINANT_DEFECT_TYPE']
all_cols  = [f.name for f in training_df.schema.fields]
input_cols = [c for c in all_cols if c.upper() not in [x.upper() for x in EXCLUDE_COLS]]
label_col  = 'YIELD_GOOD'

In [ ]:
# ============================================================================
# SCALE CLUSTER FOR DDP
# ============================================================================
from snowflake.ml.runtime_cluster import scale_cluster, get_nodes

NUM_NODES = 2
print(f"Scaling cluster to {NUM_NODES} nodes for DDP...")
scale_cluster(NUM_NODES)
nodes = get_nodes()
print(f"Active nodes: {len(nodes)}")
for i, n in enumerate(nodes):
    print(f"   Node {i}: {n['name']}  cpus={n['cpus']}")

print(f"\nML Dataset: {DATASET_NAME}")
print(f"Feature columns: {len(input_cols)}")
print(f"Label column: {label_col}")
print(f"Features: {input_cols[:5]}")

---

## 📘 Section 2 — Model Architecture

The DNN is defined **inside** the training function in Section 3 — not at the notebook scope. This is required for DDP: worker processes need to be able to re-import and re-build the class from scratch, so the definition must travel with the training function.

If you want to reuse the architecture across notebooks, move it to a Python module on the stage and import it inside the training function.


---

## 📘 Section 3 — DDP Training Function

The training function runs on **each** distributed worker. Essential patterns:

1. **Import inside the function** — workers get a clean module namespace
2. **`get_context()`** — returns rank, local_rank, dataset_map, model_dir
3. **`dist.init_process_group()`** — initialize the collective
4. **`DDP(model)`** — wrap the model so gradients synchronize automatically
5. **`get_shard()`** — each worker receives its unique data partition
6. **LR scaling** — multiply learning rate by `world_size` (standard DDP recipe)
7. **Save + eval on rank 0 only** — prevents duplicate writes and avoids double-counting metrics


In [ ]:
# ============================================================================
# MODEL DEFINITION (shared by training function + registry cell)
# ============================================================================
class WaferYieldDNN(nn.Module):
    def __init__(self, input_size, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, hidden), nn.ReLU(), nn.BatchNorm1d(hidden), nn.Dropout(0.3),
            nn.Linear(hidden, hidden // 2), nn.ReLU(), nn.BatchNorm1d(hidden // 2), nn.Dropout(0.2),
            nn.Linear(hidden // 2, 1), nn.Sigmoid(),
        )
    def forward(self, x): return self.net(x)

# ============================================================================
# DDP TRAINING FUNCTION (evaluates on held-out test set, rank 0 only)
# ============================================================================
INPUT_COLS = input_cols
LABEL_COL  = label_col
INPUT_SIZE = len(input_cols)
KEEP_COLS  = set(input_cols + [label_col])

def train_ddp_func():
    import os, torch, torch.nn as nn, torch.optim as optim, torch.distributed as dist
    import numpy as np
    from torch.nn.parallel import DistributedDataParallel as DDP
    from torch.utils.data import DataLoader
    from snowflake.ml.modeling.distributors.pytorch import get_context

    context = get_context()
    rank = context.get_rank()
    local_rank = context.get_local_rank()
    world_size = context.get_world_size()

    backend = "nccl" if torch.cuda.is_available() else "gloo"
    dist.init_process_group(backend=backend)
    device = torch.device(f"cuda:{local_rank}" if torch.cuda.is_available() else "cpu")
    print(f"[Rank {rank}/{world_size}] device={device}")

    class _WaferYieldDNN(nn.Module):
        def __init__(self, input_size, hidden=128):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(input_size, hidden), nn.ReLU(), nn.BatchNorm1d(hidden), nn.Dropout(0.3),
                nn.Linear(hidden, hidden // 2), nn.ReLU(), nn.BatchNorm1d(hidden // 2), nn.Dropout(0.2),
                nn.Linear(hidden // 2, 1), nn.Sigmoid(),
            )
        def forward(self, x): return self.net(x)

    def numeric_collate(batch):
        if isinstance(batch, dict):
            return {k: torch.as_tensor(v) for k, v in batch.items() if k in KEEP_COLS and not np.issubdtype(getattr(v, 'dtype', type(v)), np.datetime64)}
        return batch

    model = _WaferYieldDNN(INPUT_SIZE)
    if torch.cuda.is_available():
        model = nn.SyncBatchNorm.convert_sync_batchnorm(model)
    model = model.to(device)
    model = DDP(model, device_ids=[local_rank] if torch.cuda.is_available() else None)

    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001 * world_size)

    dataset_map = context.get_dataset_map()
    train_ds = dataset_map["train"].get_shard().to_torch_dataset(batch_size=1024)
    train_dl = DataLoader(train_ds, batch_size=None, collate_fn=numeric_collate)

    EPOCHS = 25
    model.train()
    for epoch in range(EPOCHS):
        epoch_loss, num_batches = 0.0, 0
        for batch_dict in train_dl:
            if not batch_dict: continue
            X = torch.stack([batch_dict[c].squeeze() for c in INPUT_COLS], dim=1).float().to(device)
            y = batch_dict[LABEL_COL].squeeze().float().to(device)
            optimizer.zero_grad()
            loss = criterion(model(X).squeeze(), y)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item(); num_batches += 1
        if rank == 0 and (epoch + 1) % 5 == 0:
            print(f"   Epoch {epoch+1}/{EPOCHS} avg_loss={epoch_loss/max(num_batches,1):.4f}")

    if rank == 0 and "test" in dataset_map:
        from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
        model.eval()
        y_true, y_prob = [], []
        test_dl = DataLoader(dataset_map["test"].get_shard().to_torch_dataset(batch_size=1024), batch_size=None, collate_fn=numeric_collate)
        with torch.no_grad():
            for b in test_dl:
                if not b: continue
                Xb = torch.stack([b[c].squeeze() for c in INPUT_COLS], dim=1).float().to(device)
                yb = b[LABEL_COL].squeeze().float().to(device)
                pb = model(Xb).squeeze().detach().cpu().numpy().flatten()
                y_true.extend(yb.cpu().numpy().flatten().tolist())
                y_prob.extend(pb.tolist())
        y_pred = [1 if p >= 0.5 else 0 for p in y_prob]
        metrics = {
            "test_accuracy":  accuracy_score(y_true, y_pred),
            "test_precision": precision_score(y_true, y_pred, zero_division=0),
            "test_recall":    recall_score(y_true, y_pred, zero_division=0),
            "test_f1":        f1_score(y_true, y_pred, zero_division=0),
            "test_roc_auc":   roc_auc_score(y_true, y_prob) if len(set(y_true)) > 1 else float("nan"),
        }
        print(f"[Rank 0] TEST METRICS: {metrics}")

        model_dir = context.get_model_dir()
        torch.save(model.module.state_dict(), os.path.join(model_dir, "wafer_yield_ddp_model.pt"))
        print(f"[Rank 0] Model saved to: {model_dir}/wafer_yield_ddp_model.pt")

    dist.destroy_process_group()
    print(f"[Rank {rank}] training complete")

print("Training function defined (DDP, LR-scaled, test eval on rank 0)")

---

## 📘 Section 4 — Configure and Launch Distributed Training

Use `PyTorchDistributor` with `PyTorchScalingConfig` to configure the distributed training job.

In [ ]:
# ============================================================================
# CONFIGURE PYTORCHDISTRIBUTOR
# ============================================================================

NUM_WORKERS_PER_NODE = 1
NUM_GPUS_PER_WORKER = 1

scaling_config = PyTorchScalingConfig(
    num_nodes=NUM_NODES,
    num_workers_per_node=NUM_WORKERS_PER_NODE,
    resource_requirements_per_worker=WorkerResourceConfig(
        num_cpus=0,
        num_gpus=NUM_GPUS_PER_WORKER,
    ),
)

pytorch_trainer = PyTorchDistributor(
    train_func=train_ddp_func,
    scaling_config=scaling_config,
)

print("PyTorchDistributor configured")
print(f"   Nodes: {NUM_NODES}")
print(f"   Workers per node: {NUM_WORKERS_PER_NODE}")
print(f"   GPUs per worker: {NUM_GPUS_PER_WORKER}")
print(f"   Total GPUs: {NUM_NODES * NUM_WORKERS_PER_NODE * NUM_GPUS_PER_WORKER}")

In [ ]:
# ============================================================================
# RUN DISTRIBUTED TRAINING
# ============================================================================

print("🚀 Starting distributed DDP training...")
print("=" * 60)

# Run the distributed training job
# Pass the data_connector via dataset_map
response = pytorch_trainer.run(
    dataset_map={'train': sharded_train, 'test': sharded_test}
)

print("=" * 60)
print("✅ Distributed training complete!")


---

## 📘 Section 5 — Retrieve Trained Model

For multi-node DDP, the model is automatically synchronized to a Snowflake stage. Use `get_model_dir()` from the response to locate it.

In [ ]:
# ============================================================================
# RETRIEVE MODEL FROM RESPONSE
# ============================================================================

# Get the model directory from the training response (this is a stage path)
model_dir = response.get_model_dir()
print(f"📁 Model stage location: {model_dir}")
print(f"✅ Model saved to Snowflake stage and ready for registry")


---

## 📘 Section 6 — Multi-Node Model Persistence with Stages

For multi-node training, specify an `artifact_stage_location` to persist the model to a Snowflake stage:

```python
response = pytorch_trainer.run(
    dataset_map={'train': data_connector},
    artifact_stage_location="DB_NAME.SCHEMA_NAME.STAGE_NAME"
)

# Model saved at: DB_NAME.SCHEMA_NAME.STAGE_NAME/model/{request_id}/
stage_location = response.get_model_dir()
```

This ensures the model is accessible across nodes and persisted beyond the training session.

In [ ]:
# ============================================================================
# REGISTER MODEL TO SNOWFLAKE MODEL REGISTRY
# ============================================================================

from snowflake.ml.registry import Registry
import pandas as pd

# Get the model stage path from training response
model_dir = response.get_model_dir()
stage_model_path = f"{model_dir}/wafer_yield_ddp_model.pt"

# Recreate model architecture and load from stage
trained_model = WaferYieldDNN(input_size=len(input_cols))

# Download temporarily just to load state dict
import tempfile
local_temp_dir = tempfile.mkdtemp()
session.file.get(stage_model_path, local_temp_dir)

import glob
downloaded_files = glob.glob(os.path.join(local_temp_dir, "*wafer_yield_ddp_model.pt*"))
if not downloaded_files:
    downloaded_files = glob.glob(os.path.join(local_temp_dir, "*.pt"))
    
trained_model.load_state_dict(torch.load(downloaded_files[0]))
trained_model.eval()

print(f"✅ Model loaded from stage: {stage_model_path}")
print(f"   Parameters: {sum(p.numel() for p in trained_model.parameters()):,}")

# Create registry
registry = Registry(session=session)

# Create sample input for signature inference
sample_input = pd.DataFrame({col: [0.0] for col in input_cols}).astype('float32')

# Register the model
mv = registry.log_model(
    model=trained_model,
    model_name="WAFER_YIELD_DDP_MODEL",
    version_name=f"v_{datetime.now().strftime('%Y%m%d_%H%M%S')}",
    sample_input_data=sample_input,
    options={
        "embed_local_ml_library": True
    }
)

print(f"✅ Model registered to Snowflake Model Registry")
print(f"   Name: {mv.model_name}")
print(f"   Version: {mv.version_name}")

# Clean up temp directory
import shutil
shutil.rmtree(local_temp_dir)


---

## 📘 Summary

### What We Covered

| Topic | Snowflake API |
|---|---|
| **Sharded data loading** | `ShardedDataConnector.from_dataset()` |
| **Training orchestration** | `PyTorchDistributor(train_func, scaling_config)` |
| **Resource config** | `PyTorchScalingConfig(num_nodes, num_workers_per_node, ...)` |
| **Rank / device / datasets** | `get_context()` inside the training function |
| **Model persistence** | `artifact_stage_location` → Snowflake stage |
| **Registry** | `registry.log_model()` for downstream deployment |

### Key Takeaways

1. **No cluster management** — Snowflake handles all orchestration
2. **Standard PyTorch code** — DDP logic is portable
3. **Automatic data sharding** — each worker gets a unique partition
4. **Gradient sync is automatic** — DDP wrapper calls `AllReduce` each step
5. **LR scales with world_size** — keeps per-sample signal-to-noise constant

### DDP Decision Aid

| ✅ DDP helps | ❌ DDP won't help |
|---|---|
| Per-epoch training time is slow | Training is already fast |
| Multiple GPUs available | Only 1 GPU available |
| Single-GPU utilization is high | Single-GPU utilization is low (data-bottlenecked) |
| Model fits on one GPU | Model doesn't fit on one GPU (use FSDP instead) |

> **Tip**: Profile with `nvidia-smi` and Snowflake's compute-pool monitors before reaching for DDP.


In [ ]:
# ============================================================================
# END OF NOTEBOOK 2c
# ============================================================================

print("=" * 60)
print("✅ Notebook 2c Complete: Distributed Data Parallel (DDP)")
print("=" * 60)
print()
print("📊 Key APIs Used:")
print("   • ShardedDataConnector.from_dataframe() — Data loading")
print("   • PyTorchDistributor — Distributed training orchestration")
print("   • PyTorchScalingConfig — Resource configuration")
print("   • get_context() — Rank, device, dataset access in workers")
print()
print("🚀 DDP enables linear scaling across GPUs for large workloads")
print()
print("➡️  For even larger models, explore FSDP (Fully Sharded Data Parallel)")